In [ ]:
from datetime import datetime
from pathlib import Path
import sys

sys.path.insert(0, str(Path("../..").resolve()))

import ad_safe_lib as ad_safe

print("challenge:", ad_safe.CHALLENGE_DIR)
print("device:", ad_safe.DEVICE)
print("dataset:", ad_safe.DATA_DIR)
examples_root = ad_safe.AD_SAFE_RUNS_DIR / "notebook_examples"
examples_root.mkdir(parents=True, exist_ok=True)


# Staged Pretrained Training With Adversarial Augmentation

Three small phases: classifier/head on top of a pretrained backbone, then the full model, then the full model with adversarial dataset augmentation. The final comparison uses the same correctly classified sample for every phase and estimates the minimum epsilon that flips each checkpoint.


In [ ]:
run_id = "notebook-02-" + datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
output_dir = examples_root / run_id
train_source = ad_safe.DatasetSourceSpec("train", fraction=0.02, seed=20)
eval_sources = {
    "val": ad_safe.DatasetSourceSpec("val", fraction=0.05, seed=101),
    "test": ad_safe.DatasetSourceSpec("test", fraction=0.05, seed=102),
}

phase_configs = [
    (
        "head",
        ad_safe.TrainingConfig(
            base_model="mobilenet_v3_small",
            epochs=1,
            patience=1,
            batch_size=8,
            learning_rate=(1e-3,),
            resplit_runs=1,
        ),
        False,
        0,
    ),
    (
        "full",
        ad_safe.TrainingConfig(
            base_model="mobilenet_v3_small",
            epochs=1,
            patience=1,
            batch_size=8,
            learning_rate=(2e-5,),
            resplit_runs=1,
        ),
        True,
        0,
    ),
    (
        "full_adv",
        ad_safe.TrainingConfig(
            base_model="mobilenet_v3_small",
            epochs=2,
            patience=2,
            batch_size=8,
            learning_rate=(1e-5,),
            resplit_runs=1,
        ),
        True,
        0,
    ),
]
phases = tuple(
    ad_safe.PhaseSpec(
        job_index=0,
        phase_index=index,
        prefix=f"mobilenet_{name}",
        title=name,
        requested_seed=10 + index,
        config=config,
        unfreeze_all=unfreeze_all,
        unfreeze_top=unfreeze_top,
        model_filename=f"mobilenet_{name}.pt",
        history_filename=f"mobilenet_{name}_history.png",
        json_filename=f"mobilenet_{name}.json",
    )
    for index, (name, config, unfreeze_all, unfreeze_top) in enumerate(phase_configs)
)

# Adversarial enrichment via EnrichmentJobSpec
# Note: This enrichment applies to all training phases within the job
adversarial_enrichment_jobs = (
    ad_safe.EnrichmentJobSpec(
        phases=(
            ad_safe.EnrichmentPhaseSpec(
                strategy=ad_safe.AdversarialStrategy(epsilon=0.08, steps=2)
            ),
        )
    ),
)

plan = ad_safe.RunPlan(
    output_dir=output_dir,
    run_id=run_id,
    train_split="train",
    eval_splits=("val", "test"),
    jobs=(
        ad_safe.JobSpec(
            job_index=0,
            job_id="mobilenet_v3_small_staged",
            title="mobilenet staged",
            backbone="mobilenet_v3_small",
            phases=phases,
            enrichment_jobs=adversarial_enrichment_jobs,
        ),
    ),
    resume=False,
    force=True,
    setup_path=output_dir / "setup.json",
    check_foreign_contract=False,
    train_source=train_source,
    eval_sources=eval_sources,
)

result = ad_safe.run_training_plan(plan)
final_phase = result.phase_results[-1]
final_model_path = final_phase.model_path
[(phase.prefix, phase.model_path) for phase in result.phase_results]


In [ ]:
rows = [
    ad_safe.ModelEvalSpec(path=phase.model_path, title=phase.prefix)
    for phase in result.phase_results
]
ad_safe.run_evaluation_plan(
    ad_safe.EvaluationPlan(
        models=tuple(rows),
        datasets=(
            ad_safe.DatasetEvalSpec(name="val", batch_size=8, source=eval_sources["val"]),
            ad_safe.DatasetEvalSpec(name="test", batch_size=8, source=eval_sources["test"]),
        ),
        output_dir=output_dir,
        sort_key="acc_test",
        title="Staged phase comparison",
    )
)


In [ ]:
from IPython.display import display
import torch

test_dataset = ad_safe.load_dataset_source(eval_sources["test"])
class_names = ad_safe.get_dataset_classes(test_dataset)
phase_models = [
    (phase, ad_safe.load_model(phase.model_path).eval())
    for phase in result.phase_results
]

comparison_sample_index = 0
found_all_correct = False
for index in range(min(128, len(test_dataset))):
    image_tensor, label_idx = test_dataset[index]
    true_class_name = class_names[label_idx]
    true_label = ad_safe.CLASS_NAMES.index(true_class_name)
    input_tensor = image_tensor.unsqueeze(0).to(ad_safe.DEVICE)
    predictions = []
    with torch.inference_mode():
        for _, phase_model in phase_models:
            logits = ad_safe.forward_logits(phase_model, input_tensor)
            predictions.append(int(logits.argmax(dim=1).item()))
    if all(prediction == true_label for prediction in predictions):
        comparison_sample_index = index
        found_all_correct = True
        break

print("comparison sample index:", comparison_sample_index)
print("correct for every phase:", found_all_correct)
for _, phase_model in phase_models:
    del phase_model
ad_safe.release_torch_memory()


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from torch import nn
import torch

image_tensor, label_idx = test_dataset[comparison_sample_index]
true_class_name = class_names[label_idx]
true_label = ad_safe.CLASS_NAMES.index(true_class_name)
original_tensor = image_tensor.unsqueeze(0).to(ad_safe.DEVICE)
criterion = nn.CrossEntropyLoss()
max_epsilon = 0.16
attack_steps = 4

def attack_minimal(model):
    return ad_safe.generate_adversarial_perturbation(
        model=model,
        x_original=original_tensor,
        criterion=criterion,
        target_labels=torch.tensor([true_label], device=ad_safe.DEVICE),
        strategy=ad_safe.MinimalFlipPgdStrategy(
            max_epsilon=max_epsilon,
            num_steps=attack_steps,
            search_steps=10,
            refinement_steps=6,
        ),
    )

fig, axes = plt.subplots(1, 1 + len(result.phase_results), figsize=(4.2 * (1 + len(result.phase_results)), 4.4))
axes[0].imshow(image_tensor.permute(1, 2, 0).numpy())
axes[0].set_title(f"Original\nTrue: {true_class_name}")
axes[0].axis("off")

for axis, phase in zip(axes[1:], result.phase_results):
    phase_model = ad_safe.load_model(phase.model_path)
    phase_model.eval()
    attack = attack_minimal(phase_model)
    epsilon_text = f"{attack.epsilon:.4f}" if attack.attack_success else f">{max_epsilon:.2f}"

    axis.imshow(attack.adversarial_tensor[0].detach().cpu().permute(1, 2, 0).numpy())
    axis.set_title(
        f"{phase.phase_title}\n"
        f"orig: {ad_safe.CLASS_NAMES[attack.original_prediction]} {attack.original_confidence:.2f}\n"
        f"adv: {ad_safe.CLASS_NAMES[attack.prediction]} {attack.confidence:.2f}\n"
        f"min_eps={epsilon_text} Linf={attack.linf:.4f}\n"
        f"RMS={attack.rms:.4f} MAE={attack.mae:.4f}"
    )
    axis.axis("off")
    print(
        phase.phase_title,
        "orig=", ad_safe.CLASS_NAMES[attack.original_prediction], f"{attack.original_confidence:.3f}",
        "adv=", ad_safe.CLASS_NAMES[attack.prediction], f"{attack.confidence:.3f}",
        "true_conf:", f"{attack.original_true_confidence:.3f}->{attack.true_confidence:.3f}",
        "min_eps=", epsilon_text,
        "linf=", f"{attack.linf:.4f}",
        "rms=", f"{attack.rms:.4f}",
        "mae=", f"{attack.mae:.4f}",
    )
    del phase_model
    ad_safe.release_torch_memory()

fig.suptitle(
    f"Same sample adversarial comparison | minimal perturbation | max_epsilon={max_epsilon} steps={attack_steps} | sample={comparison_sample_index}"
)
fig.tight_layout()
ad_safe.save_figure(fig, output_dir / "phase_adversarial_comparison.png")
display(fig)
plt.close(fig)
